# Building a Handwritten Digit Recognition Model

This model could identify handwritten digits in more realistic scenarios. In order to adjust for specific scenarios we have used **SVHN (Street View House Numbers)** dataset which contains house numbers in Google Street View images which reflects real world lighting and positioning.

# Data Preprocessing

In [5]:
import torch
import torchvision
import torch.nn as nn
import torchvision.transforms.v2 as transforms
from torch.utils.data import DataLoader

# Modern 2026 data augmentation strategy
transform = transforms.Compose([
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.RandomGrayscale(p=0.2), # Some digits might be color-coded
    transforms.RandomRotation(degrees=15),
    transforms.Normalize((0.4377, 0.4438, 0.4728), (0.1980, 0.2010, 0.1970))
])

train_set = torchvision.datasets.SVHN(root='../data', split='train', download=True, transform=transform)
test_set = torchvision.datasets.SVHN(root='../data', split='test', download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=4)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False)

# Building The Model (Deformable Model Rather Than Traditional CNN)

Instead of a standard nn.Conv2d, we use an offset-based convolution. This allows the model to "look" at the specific curves of a "2" or "8" regardless of how slanted they are.

In [6]:
import torchvision.ops as ops

class DeformableNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        
        # Deformable Layer setup
        self.offset_conv = nn.Conv2d(32, 18, kernel_size=3, padding=1) # 18 = 2 * kernel_size^2
        self.deform_conv = ops.DeformConv2d(32, 64, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 16 * 16, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        
        # Calculate offsets then apply Deformable Conv
        offsets = self.offset_conv(x)
        x = torch.relu(self.deform_conv(x, offsets))
        
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

# Training With Modern Schedulers

We'll use the OneCycleLR scheduler, which helps the model find a stable local minimum faster.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeformableNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1) # Prevents over-fitting

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.01, 
                                                steps_per_epoch=len(train_loader), epochs=10)

for epoch in range(10):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

# Evaluation And Error Analysis

Using confusion matrix to analyze how the model confuses to some specific digits

In [11]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds))
# Look specifically for the "F1-Score" per digit

              precision    recall  f1-score   support

           0       0.91      0.88      0.90      1744
           1       0.92      0.95      0.93      5099
           2       0.92      0.94      0.93      4149
           3       0.85      0.87      0.86      2882
           4       0.91      0.91      0.91      2523
           5       0.89      0.89      0.89      2384
           6       0.87      0.86      0.86      1977
           7       0.93      0.88      0.90      2019
           8       0.89      0.82      0.85      1660
           9       0.83      0.87      0.85      1595

    accuracy                           0.90     26032
   macro avg       0.89      0.88      0.89     26032
weighted avg       0.90      0.90      0.90     26032



# Why this is superior
Dataset Realism: SVHN introduces noise that MNIST lacks, making your model ready for "wild" images.

Architectural Innovation: Deformable convolutions (DCN) are much more efficient at capturing handwritten "flow" than rigid CNNs.

Training Robustness: Using label_smoothing and AdamW ensures your model doesn't just memorize the training set, but learns the underlying geometry of the digits.